# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset basic information
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id
record_sets = dataset.record_sets
print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields (@id):")
    for f in fields:
        f_id = f['@id'] if isinstance(f, dict) else f
        print(f"    - {f_id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract all (non-documentation) record sets
from collections.abc import Iterable

user_record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in user_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    # flatten nested structures if needed
    if records:
        # Ensure records are properly converted if they are not dicts
        if not isinstance(records[0], dict):
            records = [dict(r) for r in records]
        dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for the first available record set
if user_record_sets:
    first_rs = user_record_sets[0]
    if first_rs in dataframes:
        print(f"DataFrame for record set '{first_rs}' loaded.")
        print(dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Explore numeric field in the protein data record set

# You may need to adjust the record set and field ids below based on Section 2 output

# For FAIR^2, one record set is likely the main protein table.
main_record_set = user_record_sets[0]  # Use the first record set
df = dataframes[main_record_set]

# Find a numeric field (by inspecting the df)
print(df.dtypes)  # For exploration, user would look for a suitable numeric field

# Suppose 'cr:peptide_count' is one such field (change as per your actual columns)
numeric_field = None
for col in df.columns:
    if (df[col].dtype == 'int64' or df[col].dtype == 'float64'):
        numeric_field = col
        break
if not numeric_field:
    raise RuntimeError("No numeric field found in the selected record set.")
print(f"Using numeric field: {numeric_field}")

# Filter rows with large values (> 10 for demonstration)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., 'cr:accession' or similar)
group_field = None
for col in df.columns:
    if df[col].dtype == 'object' and col != numeric_field:
        group_field = col
        break
if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print("Grouped data (mean) by:", group_field)
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If there is a second numeric field, display a scatter plot
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
if len(numeric_cols) >= 2:
    plt.figure(figsize=(6,6))
    xcol, ycol = numeric_cols[0], numeric_cols[1]
    plt.scatter(df[xcol], df[ycol], alpha=0.5)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.title(f"Scatter plot of {xcol} vs. {ycol}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Add your findings after the EDA & visualization steps above. -->
In this notebook, we successfully loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We examined available record sets and fields, loaded data into pandas, performed basic filtering and normalization on numeric fields, grouped data by categorical fields, and visualized key distributions. 

This approach provides a robust foundation for further biomarker discovery, protein expression studies, and advanced statistical analyses.